In [1]:
import os

In [2]:
%pwd

'c:\\Users\\dibim\\OneDrive\\Desktop\\Text_summarizer\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'c:\\Users\\dibim\\OneDrive\\Desktop\\Text_summarizer'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: Path

In [6]:
from TextSummarizationProject.constants import *
from TextSummarizationProject.utils.common import read_yaml, create_directories

In [14]:
class ConfigurationManager:
    def __init__(self, 
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artefacts_root])
    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            tokenizer_name=config.tokenizer_name
        )

        return data_transformation_config

In [15]:
import os
from TextSummarizationProject.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

In [ ]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)

    def convert_examples_to_features(self, example_batch):
        input_encodings = self.tokenizer(
            example_batch['dialogue'], 
            max_length=1024, 
            truncation=True
        )

        target_encodings = self.tokenizer(
            text_target=example_batch['summary'],
            max_length=128, 
            truncation=True
        )

        return {
            'input_ids':      input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels':         target_encodings['input_ids']
        }
    
    def convert(self):
        logger.info("Loading dataset from disk")
        dataset_samsum = load_from_disk(self.config.data_path)

        logger.info("Converting examples to features")
        dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched=True)

        logger.info("Saving transformed dataset to disk")
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir, "samsum_dataset")  )
    

In [17]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e

[2026-05-11 14:40:02,700 - INFO - common - yaml file: config\config.yaml loaded successfully]
[2026-05-11 14:40:02,704 - INFO - common - yaml file: params.yaml loaded successfully]
[2026-05-11 14:40:02,707 - INFO - common - Directory: artefacts created successfully]
[2026-05-11 14:40:02,708 - INFO - common - Directory: artefacts/data_transformation created successfully]
[2026-05-11 14:40:03,355 - INFO - _client - HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]


[2026-05-11 14:40:03,357 - WARNING - _http - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.]
[2026-05-11 14:40:03,455 - INFO - _client - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-05-11 14:40:03,546 - INFO - _client - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]


c:\Users\dibim\OneDrive\Desktop\Text_summarizer\textS\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dibim\.cache\huggingface\hub\models--google--pegasus-cnn_dailymail. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


[2026-05-11 14:40:03,863 - INFO - _client - HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-05-11 14:40:03,968 - INFO - _client - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 200 OK"]
[2026-05-11 14:40:04,048 - INFO - _client - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 200 OK"]
[2026-05-11 14:40:04,294 - INFO - _client - HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"]
[2026-05-11 14:40:04,489 - INFO - _client - HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main?recursive=true&expand

Map: 100%|██████████| 818/818 [00:00<00:00, 8804.53 examples/s]

[2026-05-11 14:40:09,900 - INFO - 3740168884 - Saving transformed dataset to disk]



Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 197407.40 examples/s]
